# v0.39.0 — The `surreal-orm-lite` command line

## 1. Intro

Since v0.39.0, `pip install surreal-orm-lite` installs a `surreal-orm-lite` script (also run as
`python -m surreal_orm_lite.cli`): the full SurrealDB-ORM's `surreal-orm` command line — `makemigrations`, `migrate`,
`upgrade`, `rollback`, `status`, `sqlmigrate`, `inspectdb`, `schemadiff` and `shell` — over v0.38.0's migrations and
v0.30.0's introspection, with the same global options and `SURREAL_*` environment variables. Exit codes: 0 success,
1 the command failed, 2 a usage error, 130 interrupted.

The commands behave the same on SurrealDB 3.x and 2.x, except where the API they call differs: the 3.x-only
migration forms are refused on 2.x, and the `shell` prints one entry per statement as the server answers it (§9). The
outputs below come from a SurrealDB 3.x server.

`main()` refuses to run inside a running event loop — a notebook's kernel is one — so the notebook runs the command
line as a user would, in a subprocess.

## 2. Connect, and a project directory

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The notebook's own
connection only prepares and removes the database `cli_demo`; every command gets its settings from the environment
(`SURREAL_URL`, `SURREAL_NAMESPACE`, `SURREAL_DATABASE`; the user and password default to `root`).

The project is a temporary directory holding `app_models.py`; the commands run there, so the migration files land in
`migrations/` and every path they print is relative.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
URL = f"ws://{HOST}:{PORT}/rpc"
SurrealDBConnectionManager.set_connection(
    url=URL,
    user="root",
    password="root",
    namespace="examples",
    database="cli_demo",
)
client = await SurrealDBConnectionManager.get_client()
await client.query("REMOVE DATABASE IF EXISTS cli_demo; DEFINE DATABASE cli_demo;", {})
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


In [2]:
import subprocess
import sys
import tempfile
from pathlib import Path

PROJECT = Path(tempfile.mkdtemp(prefix="cli_demo_"))
ENV = {
    **os.environ,
    "SURREAL_URL": URL,
    "SURREAL_NAMESPACE": "examples",
    "SURREAL_DATABASE": "cli_demo",
    "COLUMNS": "100",  # argparse wraps --help to the terminal width
    "PYTHONDONTWRITEBYTECODE": "1",  # app_models.py changes in §7
}


def run(*args: str, stdin: str | None = None, show: bool = True, drop: tuple[str, ...] = ()) -> subprocess.CompletedProcess[str]:
    """Run ``surreal-orm-lite <args>`` in the project directory and print its stdout, its stderr and its exit
    status. Lines starting with one of *drop* are left out of the printed output."""
    done = subprocess.run(
        [sys.executable, "-m", "surreal_orm_lite.cli", *args],
        cwd=PROJECT,
        env=ENV,
        input=stdin,
        capture_output=True,
        text=True,
        check=False,
    )
    if show:
        print("$ surreal-orm-lite", " ".join(arg if arg else '""' for arg in args))
        for line in done.stdout.splitlines():
            if not line.startswith(drop):
                print(line)
        for line in done.stderr.splitlines():
            print("stderr:", line)
        print(f"[exit {done.returncode}]")
    return done


MODELS_V1 = """
from surreal_orm_lite import BaseSurrealModel, Index, SchemaMode, SurrealConfigDict


class Author(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="author", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    name: str


class Book(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="book",
        schema_mode=SchemaMode.SCHEMAFULL,
        indexes=[Index("book_title", "title")],
    )

    id: str | None = None
    title: str
    author: str
"""
(PROJECT / "app_models.py").write_text(MODELS_V1)
print(sorted(path.name for path in PROJECT.iterdir()))

['app_models.py']


## 3. `--help` and `--version`

Global options go before the command; each command has its own `--help`. The version cell prints only the first word
of `--version`'s output and whether the number is the package's `__version__`.

In [3]:
run("--help");

$ surreal-orm-lite --help
usage: surreal-orm-lite [-h] [--version] [--migrations-dir MIGRATIONS_DIR] [--url URL]
                        [--namespace NAMESPACE] [--database DATABASE] [--user USER]
                        [--password PASSWORD] [--connection CONNECTION]
                        COMMAND ...

SurrealDB-ORM-lite command line: migrations, schema introspection and a SurrealQL shell.

positional arguments:
  COMMAND
    makemigrations      Generate migration files from model changes.
    migrate             Apply pending schema migrations.
    upgrade             Apply data migrations to transform records.
    rollback            Rollback migrations to TARGET.
    status              Show migration status.
    sqlmigrate          Show SQL for MIGRATION without executing.
    inspectdb           Generate Python models from existing database tables.
    schemadiff          Compare Python models against the live database schema.
    shell               Start an interactive Surreal

In [4]:
run("makemigrations", "--help");

$ surreal-orm-lite makemigrations --help
usage: surreal-orm-lite makemigrations [-h] --name NAME [--empty] [--models MODULE]
                                       [--from-db | --no-from-db] [--drop-missing]

Generate migration files from model changes.

options:
  -h, --help            show this help message and exit
  --name, -n NAME       Migration name
  --empty               Create empty migration for manual editing
  --models, -m MODULE   Model module to import (repeatable)
  --from-db, --no-from-db
                        Diff against the live database schema (the default); --no-from-db diffs
                        against the migration files (offline)
  --drop-missing        Also emit the irreversible removals (REMOVE TABLE/FIELD/ANALYZER/...) for
                        objects that have no model
[exit 0]


In [5]:
import surreal_orm_lite

version = run("--version", show=False)
program, number = version.stdout.split()
print(program)
print("Matches surreal_orm_lite.__version__:", number == surreal_orm_lite.__version__)

surreal-orm-lite
Matches surreal_orm_lite.__version__: True


## 4. `makemigrations`: the first migration

`-m app_models` imports the module (the working directory is on `sys.path`), so its models register themselves. By
default the models are compared with the connected database — empty here — and the next file of `migrations/` is
written, with one line per operation.

In [6]:
run("makemigrations", "-n", "initial", "-m", "app_models");

$ surreal-orm-lite makemigrations -n initial -m app_models
Created migration: migrations/0001_initial.py
Operations: 6
  - Create table author
  - Add field name to author
  - Create table book
  - Add field author to book
  - Add field title to book
  - Create index book_title on book
[exit 0]


## 5. `migrate` and `status`

`migrate` applies the pending migrations, each in one SurrealDB transaction with its record in
`_surreal_orm_migrations`; `status` lists every file, applied or not.

In [7]:
run("migrate")
run("status");

$ surreal-orm-lite migrate
Applied 1 migration(s):
  - 0001_initial
[exit 0]


$ surreal-orm-lite status
Migration status:
------------------------------------------------------------
[X] 0001_initial (6 ops) [R-]
------------------------------------------------------------
Legend: [X]=applied, R=reversible, D=has data migrations
[exit 0]


## 6. `sqlmigrate`: the SurrealQL of a migration

Offline (no connection): the statements of `0001_initial`, rendered for SurrealDB 3.x by default — `--line 2` renders
them for 2.x, `--backwards` shows those that undo the migration.

In [8]:
run("sqlmigrate", "0001_initial");

$ surreal-orm-lite sqlmigrate 0001_initial
DEFINE TABLE `author` TYPE NORMAL SCHEMAFULL;

DEFINE FIELD `name` ON `author` TYPE string;

DEFINE TABLE `book` TYPE NORMAL SCHEMAFULL;

DEFINE FIELD `author` ON `book` TYPE string;

DEFINE FIELD `title` ON `book` TYPE string;

DEFINE INDEX `book_title` ON `book` FIELDS `title`;
[exit 0]


In [9]:
run("sqlmigrate", "0001_initial", "--backwards");

$ surreal-orm-lite sqlmigrate 0001_initial --backwards
REMOVE INDEX IF EXISTS `book_title` ON `book`;

REMOVE FIELD IF EXISTS `title` ON `book`;

REMOVE FIELD IF EXISTS `author` ON `book`;

REMOVE TABLE IF EXISTS `book`;

REMOVE FIELD IF EXISTS `name` ON `author`;

REMOVE TABLE IF EXISTS `author`;
[exit 0]


## 7. `schemadiff`, then a model change

`schemadiff` compares the models with the database, read-only: in sync right after `migrate`. A new optional field on
`Book` is then a difference, which `makemigrations` writes as `0002_add_field` and `migrate` applies.

In [10]:
run("schemadiff", "-m", "app_models");

$ surreal-orm-lite schemadiff -m app_models
Models and database are in sync.
[exit 0]


In [11]:
(PROJECT / "app_models.py").write_text(
    MODELS_V1.replace("    author: str\n", "    author: str\n    year: int | None = None\n")
)
run("schemadiff", "-m", "app_models")
run("makemigrations", "-n", "add_field", "-m", "app_models")
run("migrate")
run("schemadiff", "-m", "app_models");

$ surreal-orm-lite schemadiff -m app_models
Found 1 difference(s):
  - Add field year to book
[exit 0]


$ surreal-orm-lite makemigrations -n add_field -m app_models
Created migration: migrations/0002_add_field.py
Operations: 1
  - Add field year to book
[exit 0]


$ surreal-orm-lite migrate
Applied 1 migration(s):
  - 0002_add_field
[exit 0]


$ surreal-orm-lite schemadiff -m app_models
Models and database are in sync.
[exit 0]


## 8. `inspectdb`: models from the database

The models of the database's tables, generated by `generate_models_from_db()` — printed here (the first lines), or
written with `-o models.py`; `--table` reads only the named tables.

In [12]:
inspected = run("inspectdb", "--table", "book", show=False)
print("$ surreal-orm-lite inspectdb --table book")
print("\n".join(inspected.stdout.splitlines()[:25]))
print(f"[exit {inspected.returncode}]")

$ surreal-orm-lite inspectdb --table book
"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from surreal_orm_lite import BaseSurrealModel, SchemaMode, SurrealConfigDict


class Book(BaseSurrealModel):
    # Indexes: book_title
    model_config = SurrealConfigDict(table_name="book", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    author: str
    title: str
    year: int | None = None
[exit 0]


## 9. `shell`: a SurrealQL console

One request per line, piped here; `exit` ends the session. The `Connected to …` line, which shows the URL, is left out
of the printed output, and the piped lines are not echoed: each answer follows its `surreal>` prompt. A line of several statements prints one block per statement, under `-- [i/N] --` headers. On
SurrealDB 3.x, `BEGIN` and `COMMIT` are entries of their own (three blocks below); on 2.x the server answers only for
the statements inside the transaction, so the line prints the created record alone.

In [13]:
run(
    "shell",
    stdin="RETURN 1 + 1;\nBEGIN; CREATE author:x SET name = 'tx'; COMMIT;\nSELECT * FROM author;\nexit\n",
    drop=("Connected to",),
);

$ surreal-orm-lite shell
Namespace: examples, Database: cli_demo
Type 'exit' or 'quit' to exit. Enter SurrealQL queries:
------------------------------------------------------------
surreal> 2
surreal> -- [1/3] --
(empty result)
-- [2/3] --
{'id': RecordID(table_name=author, record_id='x'), 'name': 'tx'}
-- [3/3] --
(empty result)
surreal> {'id': RecordID(table_name=author, record_id='x'), 'name': 'tx'}
surreal> Goodbye!
[exit 0]


## 10. `rollback` and `status`

`rollback TARGET` undoes the migrations applied after `TARGET`, newest first; `""` undoes every one.

In [14]:
run("rollback", "0001_initial")
run("rollback", "")
run("status");

$ surreal-orm-lite rollback 0001_initial
Rolled back 1 migration(s):
  - 0002_add_field
[exit 0]


$ surreal-orm-lite rollback ""
Rolled back 1 migration(s):
  - 0001_initial
[exit 0]


$ surreal-orm-lite status
Migration status:
------------------------------------------------------------
[ ] 0001_initial (6 ops) [R-]
[ ] 0002_add_field (1 ops) [R-]
------------------------------------------------------------
Legend: [X]=applied, R=reversible, D=has data migrations
[exit 0]


## 11. Cleanup

The demo database and the project directory.

In [15]:
import shutil

await client.query("REMOVE DATABASE IF EXISTS cli_demo;", {})
await SurrealDBConnectionManager.close_connection()
shutil.rmtree(PROJECT)
print("Cleaned up:", not PROJECT.exists())

Cleaned up: True
